# Práctica 4 - Aprendizaje Supervisado II: Regresión

**Materia:** Introducción al Aprendizaje Automático  
**Tema:** Aplicacion de fundamentos matematicos, probabilidad y estadistica a traves de un piplenie de AA

**Nombre y apellido:**  
**Fecha de entrega:**  Miércoles 29/04/2026, 23:59

---

## Objetivos

Este cuadernillo tiene como objetivo simular un trabajo real en donde se deban aplicar y comparar distintos modelos de regresión sobre un dataset crudo, integrando conceptos vistos en clase:

- exploración de datos,
- partición en entrenamiento / validación,
- estandarización,
- modelos de regresión con TensorFlow,
- ajuste de hiperparámetros,
- comparación de modelos,
- evaluación con métricas.


## Instrucciones

1. Hacé una copia personal de este cuaderno antes de editarlo.
2. Completá todas las celdas pedidas.
3. No borres las celdas de enunciado.
4. Ejecutá el cuaderno de principio a fin antes de entregar.
5. Dejá visibles los resultados y gráficos.
6. Entregá el archivo `.ipynb` con el nombre indicado por la cátedra.


## Parte 0 - Cargar librerías y Tarea

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [ ]:
#Para montar tu drive así podés subir los archivos después
from google.colab import drive
drive.mount('/content/drive')


## Contexto

Supongamos que trabajás como especialista en Aprendizaje Automático en una empresa proveedora de materiales para sitios de construcción.

Para abaratar costos y acelerar el análisis de laboratorio, el jefe de la empresa te pide entrenar un modelo que permita **predecir la resistencia a compresión del concreto** a partir de distintas mediciones de sus componentes y condiciones de preparación.

La empresa te entrega un dataset llamado **`concreto_entrenar.csv`**, con mediciones reales, pero te advierte dos cosas importantes:

1. **Los datos no fueron limpiados ni analizados previamente.**  
   Por lo tanto, antes de entrenar modelos, debés inspeccionar el dataset y tomar las decisiones de preprocesamiento que consideres necesarias.

2. La empresa te entrega un nuevo archivo llamado **`concreto_nuevo.csv`**, que contiene nuevas mediciones pero **sin la columna `resistencia_compresion`**.  
   Tu tarea final será usar el mejor modelo entrenado para predecir esa variable y devolver el archivo con las predicciones.

---

## Variables disponibles

Las mediciones del dataset corresponden a:

- **`cemento`**: cantidad de cemento en la mezcla
- **`escoria_de_horno`**: cantidad de escoria de horno
- **`ceniza_volante`**: cantidad de ceniza volante
- **`agua`**: cantidad de agua
- **`superplastificante`**: cantidad de superplastificante
- **`agregado_grueso`**: cantidad de agregado grueso
- **`agregado_fino`**: cantidad de agregado fino
- **`edad_dias`**: antigüedad de la muestra en días
- **`cemento_correlacionado`**: variable sintética agregada al dataset
- **`alumno_cargador`**: variable categórica que indica quién cargó la medición
- **`resistencia_compresion`**: variable objetivo a predecir

---

## Objetivo

Debés construir un pipeline de trabajo completo que incluya:

- carga e inspección del dataset,
- análisis y limpieza,
- división en entrenamiento y validación,
- comparación de múltiples modelos de regresión,
- búsqueda de hiperparámetros con **k-fold cross validation**,
- selección del mejor modelo,
- predicción sobre nuevos datos.

---

## Entrega al jefe

La entrega final debe consistir en:

1. una breve justificación del modelo elegido,
2. una explicación de la métrica principal seleccionada,
3. un archivo CSV con las predicciones de `resistencia_compresion` para `concreto_nuevo.csv`.

---

## Importante

No existe una única respuesta correcta.  
Se espera que tomes decisiones razonadas de preprocesamiento, selección de métricas y elección del modelo.

En particular, pensá:

- ¿Qué errores son más costosos para la empresa?
- ¿Conviene penalizar mucho los errores grandes?
- ¿Importa más el error promedio absoluto?
- ¿Sirve reportar también $R^2$ para medir qué tan bien explica el modelo los datos?

Justificá tus decisiones.

##Importante: si la celda no dice que podés cambiar algo de código o completar, no la toques


## Paso 1 — Cargar el dataset

Subí el archivo `concreto_entrenar.csv` a Colab y cargalo en un DataFrame.



In [ ]:
uploaded = files.upload()
df = pd.read_csv("concreto_entrenar.csv")
df.head()

## Paso 2 — Inspección inicial del dataset

Antes de entrenar modelos, inspeccioná:

- tamaño del dataset,
- nombres de columnas,
- tipos de datos,
- valores faltantes,
- estadísticas descriptivas,
- posibles valores extraños o absurdos.

In [ ]:
#Modificá como necesites el código de abajo

print("Dimensiones:", df.shape)
print("\nColumnas:")
print(df.columns.tolist())

df.info()

df.describe(...)

#ETC.

NameError: name 'df' is not defined

## Paso 3 — Exploración de los datos

Visualizá las distribuciones de las variables numéricas y estudiá relaciones entre variables.

Preguntas orientadoras:

- ¿Hay variables con distribuciones muy extrañas?
- ¿Hay valores extremadamente grandes o negativos que no tengan sentido físico?
- ¿Hay variables muy correlacionadas entre sí?
- ¿La variable categórica parece útil?

Te dejo algunas ayudas

In [ ]:
columnas_numericas = df.select_dtypes(include=np.number).columns.tolist()
columnas_categoricas = df.select_dtypes(exclude=np.number).columns.tolist()

print("Numéricas:", columnas_numericas)
print("Categóricas:", columnas_categoricas)

In [ ]:
df[columnas_numericas].hist(figsize=(16, 12), bins=25)
plt.tight_layout()
plt.show()

In [ ]:
#Agregá histogramas, matrices de correlación, etc.

## Paso 4 — Decisiones de preprocesamiento

A partir del análisis exploratorio, decidí:

- cómo tratar los valores faltantes,
- si eliminar o conservar columnas muy correlacionadas,
- qué hacer con la variable categórica `alumno_cargador`,
- cómo tratar outliers evidentes,
- si conviene escalar las variables.

**Importante:** no hagas cambios arbitrarios. Toda decisión debe poder justificarse.

Si eliminás alguna columna/fila detallá la razón debajo

_Reemplazá este texto por tu respuesta._

In [ ]:
#Tu código limpiando los datos aquí

## Paso 5 — División en entrenamiento y validación

Separamos entre variables predictoras y el objetivo.

Después dividí el dataset disponible en:

- conjunto de entrenamiento
- conjunto de validación

Usaremos el conjunto de entrenamiento para búsqueda de hiperparámetros con validación cruzada, y el conjunto de validación para una comparación final entre modelos.

Nota: no tiene sentido que tengas conjunto de test.

In [ ]:

X = df.drop(columns=["resistencia_compresion"])
y = df["resistencia_compresion"]

In [ ]:
#Modificá como necesites el código de abajo

X_train, X_val, y_train, y_val = train_test_split(
    X, y, ..., random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Validación:", X_val.shape, y_val.shape)

## Paso 6 — Construcción del preprocesamiento

Construimos un preprocesamiento para:

- imputar valores faltantes,
- escalar variables numéricas cuando corresponda,
- codificar variables categóricas cuando corresponda.

**Atención:** algunos modelos necesitan escalado más que otros.

`ColumnTransformer` nos permite aplicar distintos preprocesamientos según el tipo de variable.

- A las columnas numéricas:
  - completamos valores faltantes con la mediana,
  - estandarizamos los valores.

- A las columnas categóricas:
  - completamos faltantes con la categoría más frecuente,
  - convertimos el texto en variables binarias mediante one-hot encoding.

Esto nos permite preparar los datos de forma consistente antes de entrenar el modelo.

In [ ]:
columnas_numericas = X_train.select_dtypes(include=np.number).columns.tolist()
columnas_categoricas = X_train.select_dtypes(exclude=np.number).columns.tolist()

print("Columnas numéricas:", columnas_numericas)
print("Columnas categóricas:", columnas_categoricas)

In [ ]:
#Lamentablemente no puedo ayudarte porque no sé qué features
#decidiste dejar. Pero la idea es que sea estandarización para variables
#numéricas y que transformes variables categóricas (si tenés)

preprocesador_base = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), columnas_numericas),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), columnas_categoricas)
    ]
)

#no modifiques este código

NameError: name 'ColumnTransformer' is not defined

## Paso 7 — Métricas

Vamos a evaluar los modelos con varias métricas:

- **RMSE**: penaliza más los errores grandes y está en las mismas unidades que la variable objetivo.
- **MAE**: mide error absoluto promedio y es más robusta a outliers.
- **R²**: indica qué proporción de la variabilidad explica el modelo.

Luego deberás decidir cuál de estas métricas es más relevante para el problema de negocio.

In [ ]:
def evaluar_modelo(nombre, modelo, X_val, y_val):
    y_pred = modelo.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    return {
        "modelo": nombre,
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2
    }

## Paso 8 — Configuración de validación cruzada

Usaremos **k-fold cross validation** sobre el conjunto de entrenamiento para seleccionar hiperparámetros, tal como vimos en clase.

Podés cambiar el valor de `n_split` si querés.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)

#Paso 9 - Modelos

## Modelo 1 — Regresión lineal

Te dejo el `Pipeline` o flujo: primero procesa con el `procesador_base` que construimos arriba y después aplica el modelo.

In [ ]:
pipe_lr = Pipeline([
    ("preprocesamiento", preprocesador_base),
    ("modelo", LinearRegression())
])

pipe_lr.fit(X_train, y_train)
res_lr = evaluar_modelo("Regresión Lineal", pipe_lr, X_val, y_val)
res_lr

## Modelo 2 — Regresión polinómica

Sugerencia: probar al menos grados 2 y 3.
Pensá también en el riesgo de sobreajuste al aumentar la complejidad del modelo.

In [ ]:
#no toques este pipeline
pipe_poly = Pipeline([
    ("preprocesamiento", ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("poly", PolynomialFeatures(include_bias=False))
            ]), columnas_numericas),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore"))
            ]), columnas_categoricas)
        ]
    )),
    ("modelo", LinearRegression())
])

#podés probar otros grados acá
param_grid_poly = {
    "preprocesamiento__num__poly__degree": [2, 3]
}

#hacemos validación cruzada
grid_poly = GridSearchCV(
    pipe_poly,
    param_grid=param_grid_poly,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_poly.fit(X_train, y_train)
print("Mejores hiperparámetros:", grid_poly.best_params_)
print("Mejor CV score (RMSE negativo):", grid_poly.best_score_)

res_poly = evaluar_modelo("Regresión Polinómica", grid_poly.best_estimator_, X_val, y_val)
res_poly

## Modelo 3 — Ridge

In [ ]:
pipe_ridge = Pipeline([
    ("preprocesamiento", preprocesador_base),
    ("modelo", Ridge())
])

#grilla para hiperparámetro. Podés cambiar o dejar los que te propongo
param_grid_ridge = {
    "modelo__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
}

#validación cruzada
grid_ridge = GridSearchCV(
    pipe_ridge,
    param_grid=param_grid_ridge,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_ridge.fit(X_train, y_train)
print("Mejores hiperparámetros:", grid_ridge.best_params_)
print("Mejor CV score:", grid_ridge.best_score_)

res_ridge = evaluar_modelo("Ridge", grid_ridge.best_estimator_, X_val, y_val)
res_ridge

## Modelo 4 — Lasso

In [ ]:
pipe_lasso = Pipeline([
    ("preprocesamiento", preprocesador_base),
    ("modelo", Lasso(max_iter=10000))
])

#grilla para hiperparámetro. Podés cambiar o dejar los que te propongo
param_grid_lasso = {
    "modelo__alpha": [0.001, 0.01, 0.1, 1.0, 10.0]
}

#validación cruzada
grid_lasso = GridSearchCV(
    pipe_lasso,
    param_grid=param_grid_lasso,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_lasso.fit(X_train, y_train)
print("Mejores hiperparámetros:", grid_lasso.best_params_)
print("Mejor CV score:", grid_lasso.best_score_)

res_lasso = evaluar_modelo("Lasso", grid_lasso.best_estimator_, X_val, y_val)
res_lasso

## Modelo 6 — Random Forest

In [ ]:
preprocesador_arboles = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), columnas_numericas),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), columnas_categoricas)
    ]
)

In [ ]:
pipe_rf = Pipeline([
    ("preprocesamiento", preprocesador_arboles),
    ("modelo", RandomForestRegressor(random_state=42))
])

param_grid_rf = {
    "modelo__n_estimators": [100, 300],
    "modelo__max_depth": [None, 5, 10, 20],
    "modelo__min_samples_split": [2, 5, 10]
}

grid_rf = GridSearchCV(
    pipe_rf,
    param_grid=param_grid_rf,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)
print("Mejores hiperparámetros:", grid_rf.best_params_)
print("Mejor CV score:", grid_rf.best_score_)

res_rf = evaluar_modelo("Random Forest", grid_rf.best_estimator_, X_val, y_val)
res_rf

## Modelo 7 — Gradient Boosting

In [ ]:
pipe_gb = Pipeline([
    ("preprocesamiento", preprocesador_arboles),
    ("modelo", GradientBoostingRegressor(random_state=42))
])

param_grid_gb = {
    "modelo__n_estimators": [100, 200],
    "modelo__learning_rate": [0.01, 0.05, 0.1],
    "modelo__max_depth": [2, 3, 4]
}

grid_gb = GridSearchCV(
    pipe_gb,
    param_grid=param_grid_gb,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid_gb.fit(X_train, y_train)
print("Mejores hiperparámetros:", grid_gb.best_params_)
print("Mejor CV score:", grid_gb.best_score_)

res_gb = evaluar_modelo("Gradient Boosting", grid_gb.best_estimator_, X_val, y_val)
res_gb

## Paso 9 — Comparación final de modelos

Compará todos los modelos en el conjunto de validación.
No te quedes solo con una métrica: observá RMSE, MAE y R².

In [ ]:
resultados = pd.DataFrame([
    res_lr,
    res_poly,
    res_ridge,
    res_lasso,
    res_rf,
    res_gb
])

#Cambiar métrica para ordenar si así te parece
resultados.sort_values(by="RMSE")

## Paso 10 — Selección del mejor modelo

Elegí el mejor modelo y justificá tu decisión.

Preguntas orientadoras:

- ¿Qué modelo tuvo menor RMSE?
- ¿Qué modelo tuvo menor MAE?
- ¿Cuál mostró mejor equilibrio entre error y capacidad de generalización?
- ¿Hay evidencia de sobreajuste?
- ¿Un modelo más complejo realmente mejoró lo suficiente como para justificar su uso?

Definí en la siguiente celda cuál será tu modelo final.

In [ ]:
mejor_modelo = grid_gb.best_estimator_   # <-- cambiar según tu decisión final

## Paso 11 — Reentrenamiento final

Una vez elegido el mejor modelo e hiperparámetros, reentrenalo usando **todos los datos disponibles de `concreto_entrenar.csv`**.

In [ ]:
mejor_modelo.fit(X, y)

## Paso 12 — Cargar `concreto_nuevo.csv`

Ahora cargá el archivo con nuevas mediciones, sin la columna objetivo.

In [ ]:
uploaded = files.upload()

In [ ]:
df_nuevo = pd.read_csv("concreto_nuevo.csv")
df_nuevo.head()

## Paso 13 — Verificar compatibilidad de columnas

Antes de predecir, asegurate de que el nuevo dataset tenga las columnas esperadas.
Recordá también que debe recibir el **mismo preprocesamiento** que los datos de entrenamiento.

In [ ]:
#Tu código aquí

## Paso 14 — Predicción de resistencia a compresión

In [ ]:
#asegurate que el código de abajo esté todo OK
predicciones = mejor_modelo.predict(df_nuevo)
predicciones[:10]

In [ ]:
df_nuevo["resistencia_compresion_pred"] = predicciones
df_nuevo.head()

## Paso 15 — Exportar archivo de entrega

Guardá el dataset con las predicciones y descargalo para entregarlo.

In [ ]:
nombre_salida = "concreto_nuevo_predicciones.csv"
df_nuevo.to_csv(nombre_salida, index=False)
print(f"Archivo guardado como: {nombre_salida}")

In [ ]:
files.download(nombre_salida)

## Preguntas finales para responder en la entrega (Sólo para doctorado)


1. ¿Qué modelo elegiste finalmente? ¿Cuál fue tu criterio?
2. ¿Qué métrica consideraste más importante para este problema y por qué?
4. ¿Qué ventajas y desventajas tiene el modelo seleccionado?
5. ¿Detectaste evidencia de sobreajuste o subajuste en algunos modelos? Justificar

_Reemplazá el texto por tu respuesta aquí_

## Entrega

Antes de entregar, verificá lo siguiente:

- [ ] completé todas las consignas;
- [ ] ejecuté el cuaderno de principio a fin;
- [ ] dejé los resultados visibles;
- [ ] guardé el archivo con el nombre correcto.

Fin de la práctica.